# Mini-Project 3 — A Framework RAG with LangChain
### IT7075: Applied AI for Cybersecurity · LangChain

In Mini-Project 2 you built RAG **by hand**: your own chunker, your own Chroma calls, your own
prompt assembly, your own retrieve-then-generate function. Now you build the same thing **with
LangChain**, and the point of the exercise is to see exactly which parts the framework takes over.

Five pieces are left for you to write, one per stage of the pipeline in the langchain_rag notebook:

| | Stage | You write |
|---|---|---|
| **TODO 1** | load the documents and split them | 3 lines |
| **TODO 2** | build the Chroma vector store | 1 line |
| **TODO 3** | turn it into a retriever (`k` + score threshold) | 2 lines |
| **TODO 4** | write the prompt template | 2 lines |
| **TODO 5** | wire the LCEL chain together with `\|` | 3 lines |

Find them by searching for `###################`:

```
###################  TODO n — title  ###################
# what to do, and how
################
```

> **Runs offline.** Embeddings are the local `all-MiniLM-L6-v2` model, so retrieval never needs an
> API key. If you have a working `OPENAI_API_KEY` the last stage of the chain calls a real model;
> if you don't — or it is out of credit — the notebook substitutes an offline answer and keeps
> going. Everything you are graded on works either way.

In [1]:

import sys
print(sys.executable)


c:\Users\ethan\venvs\it7075_langchain\Scripts\python.exe


In [2]:

import langchain
import langchain_chroma
import langchain_huggingface
import langchain_text_splitters
import pandas

print("LangChain environment is ready!")


LangChain environment is ready!


## Part 1 — Load and split the documents  ·  **TODO 1**

LangChain's loaders return **`Document` objects** (text plus metadata), not plain strings — that is
the first difference from Mini-Project 2. The splitter then works on documents and hands back more
documents, carrying the metadata along for you.

> **Expected output:** `1 document -> 10 chunks`.

In [3]:

# ============================================================
# TODO 1 — Load and Split the Documents
# IT7075: Applied AI for Cybersecurity
# ============================================================

# Import required libraries
import json
import os
import warnings
import logging
import pandas as pd

# Force CPU execution to avoid potential GPU compatibility issues
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")

# Suppress unnecessary warnings and logging messages
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message="Relevance scores must be between")
logging.getLogger("langchain_core.vectorstores.base").setLevel(logging.ERROR)

# Import LangChain document loader and text splitter
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter


# ------------------------------------------------------------
# Step 1: Locate the practice knowledge base and gold set
# ------------------------------------------------------------

def course_path(name, folders=("../../code/", "../../../code/", "./")):
    for folder in folders:
        if os.path.exists(folder + name):
            return folder + name
    return name

KB_FILE = course_path("ai_framework_kb.md")

GOLDSET_FILE = (
    "framework_goldset.json"
    if os.path.exists("framework_goldset.json")
    else "../framework_goldset.json"
)


# ------------------------------------------------------------
# Step 2: Load the knowledge base using LangChain
# ------------------------------------------------------------

# TextLoader reads the Markdown file into Document objects
loader = TextLoader(KB_FILE, encoding="utf-8")
documents = loader.load()


# ------------------------------------------------------------
# Step 3: Split the document into smaller chunks
# ------------------------------------------------------------

# Configure chunk size, overlap, and separator
splitter = CharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150,
    separator="\n"
)

# Split documents while preserving their source metadata
chunks = splitter.split_documents(documents)


# ------------------------------------------------------------
# Step 4: Verify the document loading and splitting
# ------------------------------------------------------------

print(len(documents), "document ->", len(chunks), "chunks")

print("\nFirst chunk:\n", chunks[0].page_content[:200], "...")

print("\nIts metadata:", chunks[0].metadata)


1 document -> 10 chunks

First chunk:
 # NIST AI Risk Management Framework (AI RMF)
The NIST AI Risk Management Framework (AI RMF 1.0) is a voluntary framework to help organizations manage risks across the AI lifecycle. It is organized aro ...

Its metadata: {'source': './ai_framework_kb.md'}


## Part 2 — Build the vector store  ·  **TODO 2**

In Mini-Project 2 this took three steps: embed the chunks, create a collection, then add ids,
documents, embeddings and metadata. LangChain does all of it in **one call** — `Chroma.from_documents`
embeds and stores in a single line. That is the framework's whole pitch.

We use the **local** embedding model, so no API key is needed.

> **Expected output:** `stored 10 chunks`.

In [4]:

# ============================================================
# TODO 2 — Build the Chroma Vector Store
# IT7075: Applied AI for Cybersecurity
# ============================================================

# Import LangChain's embedding model and vector database
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Step 1: Initialize the local embedding model
# Converts text into 384-dimensional numerical vectors
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

# Step 2: Create the Chroma vector store
# LangChain automatically embeds and stores all 10 chunks
db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

# Step 3: Verify that the chunks were stored successfully
print("stored", db._collection.count(), "chunks")


c:\Users\ethan\venvs\it7075_langchain\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<?, ?it/s]


stored 10 chunks


## Part 3 — Make a retriever  ·  **TODO 3**

A **retriever** is a vector store wrapped in a standard interface, so anything downstream can just
call `.invoke(question)`. It is also where your two Mini-Project 2 knobs reappear under new names:

| Mini-Project 2 (by hand) | LangChain |
|---|---|
| `k` = how many chunks | `search_kwargs={"k": ...}` |
| `max_distance` — keep hits **below** it | `score_threshold` — keep hits **above** it |

Note the flip: a **distance** is small when things are similar, but LangChain's **relevance score**
is large when things are similar. Same idea, opposite direction.

> **Expected output:** 3 chunks for the NIST question, **0** for "the capital of France" — the
> refusal you built by hand in Mini-Project 2, now handled by the retriever.

In [5]:

# ============================================================
# TODO 3 — Configure the LangChain Retriever
# IT7075: Applied AI for Cybersecurity
# ============================================================

# Step 1: Define retrieval parameters
K = 3
THRESHOLD = 0.3

# Step 2: Create the retriever from the Chroma vector store
retriever = db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": K,
        "score_threshold": THRESHOLD
    }
)

# Step 3: Test an answerable and an unanswerable question
test_questions = [
    "What are the four core functions of the NIST AI Risk Management Framework?",
    "What is the capital of France?"
]

for question in test_questions:

    # Retrieve relevant document chunks
    retrieved_docs = retriever.invoke(question)

    print("Question:", question)
    print("Retrieved:", len(retrieved_docs), "chunks")

    # Display a preview of each retrieved chunk
    for i, doc in enumerate(retrieved_docs, start=1):
        print(
            f"  Chunk {i}:",
            doc.page_content[:70].strip().replace("\n", " "),
            "..."
        )

    print()


Question: What are the four core functions of the NIST AI Risk Management Framework?
Retrieved: 3 chunks
  Chunk 1: # NIST AI Risk Management Framework (AI RMF) The NIST AI Risk Manageme ...
  Chunk 2: # NIST Generative AI Profile The NIST AI RMF Generative AI Profile (NI ...
  Chunk 3: # EU AI Act The EU Artificial Intelligence Act is the first comprehens ...

Question: What is the capital of France?
Retrieved: 0 chunks



## Part 4 — The prompt template  ·  **TODO 4**

A `ChatPromptTemplate` is the reusable prompt from the langchain_basics notebook: you write it once with `{}`
placeholders, and the chain fills them in on every question. This replaces the string `.format()`
you did by hand in Mini-Project 2.

> **Expected output:** the filled-in prompt, showing your context in the system message and the
> question in the human message.

In [6]:

# ============================================================
# TODO 4 — Create the RAG Prompt Template
# IT7075: Applied AI for Cybersecurity
# ============================================================

from langchain_core.prompts import ChatPromptTemplate

# Step 1: Define the prompt template
rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a cybersecurity assistant. "
        "Answer questions using ONLY the provided context. "
        "Do not use outside knowledge or invent information. "
        "If the context supports the answer, begin with [RAG]. "
        "If the answer is missing or unsupported, respond with "
        "[LLM] I don't know.\n\n"
        "# Context\n{context}"
    ),
    (
        "human",
        "{question}"
    )
])

# Step 2: Preview the completed prompt
preview = rag_prompt.invoke({
    "context": "(pretend context here)",
    "question": "test question"
})

# Step 3: Display the system and human messages
for message in preview.to_messages():
    print("[%s] %s\n" % (
        message.type,
        message.content[:220]
    ))


[system] You are a cybersecurity assistant. Answer questions using ONLY the provided context. Do not use outside knowledge or invent information. If the context supports the answer, begin with [RAG]. If the answer is missing or u

[human] test question



## Part 5 — Wire the chain with `|`  ·  **TODO 5**

This is the part that has no equivalent in Mini-Project 2. There you wrote an `ask()` function that
called retrieve, then built the context, then called generate. In LangChain you **declare** that
flow with the pipe operator and get a single runnable object back:

```
{"context": retriever | format_docs, "question": RunnablePassthrough()}  |  prompt  |  model  |  parser
```

Read it left to right: the dict runs the retriever **and** passes the raw question through, the
prompt fills both placeholders, the model answers, and the parser turns the reply into a string.

> **Expected output:** the NIST question answered and tagged `[RAG]`; "the capital of France"
> refused and tagged `[LLM]`, because the retriever returned nothing for it.

In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda


def format_docs(docs):
    """GIVEN: turn the retrieved Documents into one block of text for {context}."""
    return "\n\n".join(d.page_content for d in docs)


def offline_answer(prompt_value):
    """GIVEN: stands in for the model when no LLM is available (a RunnableLambda,
    exactly like the ones in langchain_basics). It reads the context out of the filled-in
    prompt and quotes it, so the chain still produces a tagged answer."""
    context = prompt_value.to_messages()[0].content.split("# Context", 1)[-1].strip()
    if context == "":
        return "[LLM] I don't know -- the knowledge base does not cover this. (offline mode)"
    return "[RAG] (offline mode) From the retrieved context: " + context[:220]


def build_model():
    """GIVEN: a real chat model if a working key is available, else the offline stand-in."""
    if os.getenv("OPENAI_API_KEY"):
        try:
            from langchain_openai import ChatOpenAI
            m = ChatOpenAI(model="gpt-4o-mini", temperature=0)
            m.invoke("ping")                       # fail fast if the key has no credit
            print("using ChatOpenAI(gpt-4o-mini)")
            return m | StrOutputParser()
        except Exception as e:
            print("(LLM unavailable - %s. Using offline mode.)" % type(e).__name__)
    else:
        print("(no OPENAI_API_KEY - using offline mode.)")
    return RunnableLambda(offline_answer)


model = build_model()


###################  TODO 5 — build the LCEL chain  ###################
# WHAT: assemble `rag_chain` from the four pieces you already have, using `|`.
#
# HOW (the "Build the RAG Chain (LCEL)" cell in the langchain_rag notebook):
rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | model
)
#
#   * the DICT is the interesting bit: both entries receive the question you invoke
#     the chain with. "context" sends it through the retriever and then through
#     format_docs; "question" passes it along untouched (RunnablePassthrough()).
#     The result is {"context": "...", "question": "..."} -- exactly the two
#     placeholders your prompt needs.
#   * `model` here already includes the output parser, so you do not add one.
#
# THEN: invoke it with just a question string -- rag_chain.invoke("...?")
################

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | model
)

# ---- given: one question the KB answers, one it does not ----------------
for q in ["What are the four core functions of the NIST AI Risk Management Framework?",
          "What is the capital of France?"]:
    print("Q:", q)
    print("A:", rag_chain.invoke(q)[:300], "\n")

(no OPENAI_API_KEY - using offline mode.)
Q: What are the four core functions of the NIST AI Risk Management Framework?
A: [RAG] (offline mode) From the retrieved context: # NIST AI Risk Management Framework (AI RMF)
The NIST AI Risk Management Framework (AI RMF 1.0) is a voluntary framework to help organizations manage risks across the AI lifecycle. It is organized around four core functi 

Q: What is the capital of France?
A: [LLM] I don't know -- the knowledge base does not cover this. (offline mode) 



## Part 6 — Ask the bot, and measure it  *(no TODO — run it and read it)*

`framework_goldset.json` holds **8 questions** about the provided knowledge base: **6 it answers**
(each with a phrase that proves it) and **2 it does not**. The two counts are the same ones you
used in Mini-Project 2:

- **found** — of the 6, how many retrieved a chunk containing the evidence?
- **refused** — of the 2, how many correctly retrieved *nothing*?

> **Expected output at the langchain_rag settings (`k=3`, threshold 0.3): found 5/6, refused 1/2.**
> Two things go wrong at once, and neither is a bug in your code:
> **Q2** (MITRE ATLAS) finds nothing, and **Q8** — which the knowledge base cannot answer —
> gets context anyway. Part 7 shows why, and why you cannot fix both by tuning.

In [8]:
GOLDSET = json.load(open(GOLDSET_FILE))["questions"]
N_ANS = sum(q["answerable"] for q in GOLDSET)
N_UNANS = len(GOLDSET) - N_ANS


def score(retriever, goldset):
    """GIVEN: count found / refused, the same two numbers as Mini-Project 2."""
    found, refused, detail = 0, 0, []
    for q in goldset:
        docs = retriever.invoke(q["question"])
        text = " ".join(d.page_content for d in docs).lower()
        hit = any(p.lower() in text for p in q["evidence"])
        if q["answerable"]:
            found += 1 if hit else 0
        else:
            refused += 1 if len(docs) == 0 else 0
        detail.append({"qid": q["qid"], "answerable": q["answerable"],
                       "chunks": len(docs), "evidence_found": hit if q["answerable"] else "-"})
    return found, refused, detail


found, refused, detail = score(retriever, GOLDSET)
print("found   %d/%d" % (found, N_ANS))
print("refused %d/%d\n" % (refused, N_UNANS))
print(pd.DataFrame(detail).to_string(index=False))

found   5/6
refused 1/2

qid  answerable  chunks evidence_found
 Q1        True       3           True
 Q2        True       0          False
 Q3        True       2           True
 Q4        True       3           True
 Q5        True       3           True
 Q6        True       1           True
 Q7       False       0              -
 Q8       False       3              -


## Part 7 — Tune the threshold  *(no TODO)*

Sweep the threshold and watch the two counts fight each other.

> **Expected output:**
>
> | threshold | found | refused | what changed |
> |---|---:|---:|---|
> | 0.1 | **6/6** | 1/2 | finds everything, but Q8 still gets context |
> | 0.2 | **6/6** | 1/2 | same |
> | **0.3** *(lecture default)* | 5/6 | 1/2 | **worst of both** — loses Q2 *and* still leaks Q8 |
> | 0.4 | 5/6 | 1/2 | |
> | 0.5 | 5/6 | 1/2 | |
> | 0.6 | 4/6 | **2/2** | finally refuses Q8 — by giving up Q2 and Q3 too |
>
> **Read that table carefully: there is no good row.** No threshold gets 6/6 *and* 2/2, and the
> last cell shows you why — Q8, which your documents **cannot** answer, scores **0.505**, while
> Q2, which they answer perfectly well, scores only **0.258**. The wrong answer looks *more*
> relevant than the right one, so no single cut-off can separate them.
>
> That is the honest finding of this project, and it is worth more than a tidy result: **tuning
> cannot fix a ranking that is already wrong.** Fixing it would need better chunking, a better
> embedding model, or a reranking step — not a different number.
>
> Mini-Project 2 showed you the other half of this trade-off: there a *loose* threshold let the
> bot answer what it shouldn't. Both failures are silent, and only a question set catches either.

In [9]:
rows = []
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
    r = db.as_retriever(search_type="similarity_score_threshold",
                        search_kwargs={"k": K, "score_threshold": t})
    f, ref, _ = score(r, GOLDSET)
    rows.append({"threshold": t, "found": "%d/%d" % (f, N_ANS),
                 "refused": "%d/%d" % (ref, N_UNANS)})

results = pd.DataFrame(rows)
print(results.to_string(index=False))
results.to_csv("results.csv", index=False)
print("\nwrote results.csv")

# Why is there no good row? Compare the best score for a question the KB ANSWERS
# with the best score for one it CANNOT answer.
print("\ntop-1 relevance score per question (higher = looks more relevant):")
for q in GOLDSET:
    s = db.similarity_search_with_relevance_scores(q["question"], k=1)[0][1]
    print("   %-4s %-7s %+.3f  %s" % (q["qid"], "answer" if q["answerable"] else "ABSENT",
                                      s, q["question"][:52]))
print("\nQ8 is not in the knowledge base yet outranks Q2, which is. No cut-off fixes that.")

 threshold found refused
       0.1   6/6     1/2
       0.2   6/6     1/2
       0.3   5/6     1/2
       0.4   5/6     1/2
       0.5   5/6     1/2
       0.6   4/6     2/2

wrote results.csv

top-1 relevance score per question (higher = looks more relevant):
   Q1   answer  +0.640  What are the four core functions of the NIST AI Risk
   Q2   answer  +0.258  What is MITRE ATLAS used for?
   Q3   answer  +0.548  Name three risks from the OWASP Top 10 for LLM Appli
   Q4   answer  +0.696  What obligations does the EU AI Act place on high-ri
   Q5   answer  +0.656  What are the core elements of Google's Secure AI Fra
   Q6   answer  +0.623  What is the MAESTRO threat modeling framework used f
   Q7   ABSENT  -0.283  What is the capital of France?
   Q8   ABSENT  +0.505  What are the five core functions of the NIST Cyberse

Q8 is not in the knowledge base yet outranks Q2, which is. No cut-off fixes that.


## Part 8: rebuild your own RAG with LangChain

Everything above used the provided knowledge base, and it was practice. Now build the same system you
built in the RAG module, this time with LangChain, using the same data and the same questions.

Copy your `my_kb/` folder and your `goldset.json` from the RAG module into this project folder. Do not
change the documents or the questions: the comparison in your report depends on both systems answering
exactly the same eight questions over exactly the same text.

No code is provided from here on. Add cells below that:

1. Load your documents with a LangChain loader and split them with a text splitter.
2. Build a vector store from the chunks and wrap it in a retriever.
3. Write a prompt template and wire the chain with the pipe operator.
4. Answer your eight questions and print the transcript, including one the documents cannot answer.
5. Score the same measurements you used in the RAG module, vary one setting at a time, and save the
   results as `results_langchain.csv`.

Note as you go which of your RAG-module functions each LangChain object replaced. That list is the
heart of your report.

## Part 8 — Custom Cybersecurity SOC RAG with LangChain

This section rebuilds my original RAG system using LangChain.
It uses the same five fictional SOC knowledge-base documents
and the same eight evaluation questions as Mini-Project 2.

In [10]:

# ============================================================
# PART 8A — Custom Cybersecurity Knowledge Base
# IT7075: Applied AI for Cybersecurity
# ============================================================

from pathlib import Path
import json
from collections import Counter

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter


# ------------------------------------------------------------
# Step 1: Locate the original cybersecurity documents
# ------------------------------------------------------------

SOC_FOLDER = Path("my_kb")

soc_files = sorted(
    file for file in SOC_FOLDER.glob("*.md")
    if file.name != "SOURCES.md"
)

# ------------------------------------------------------------
# Step 2: Load all five documents using LangChain
# ------------------------------------------------------------

soc_documents = []

for file in soc_files:
    loader = TextLoader(str(file), encoding="utf-8")
    loaded_documents = loader.load()

    for doc in loaded_documents:
        doc.metadata["source"] = file.name
        soc_documents.append(doc)

print("Documents loaded:", len(soc_documents))

for doc in soc_documents:
    print(" -", doc.metadata["source"])


# ------------------------------------------------------------
# Step 3: Load the original eight evaluation questions
# ------------------------------------------------------------

with open("goldset.json", encoding="utf-8") as file:
    soc_goldset = json.load(file)["questions"]

answerable = sum(q["answerable"] for q in soc_goldset)
unanswerable = len(soc_goldset) - answerable

print("\nEvaluation questions:", len(soc_goldset))
print("Answerable:", answerable)
print("Unanswerable:", unanswerable)


# ------------------------------------------------------------
# Step 4: Verify that the original data is unchanged
# ------------------------------------------------------------

assert len(soc_documents) == 5
assert len(soc_goldset) == 8
assert answerable == 6
assert unanswerable == 2

source_texts = {
    doc.metadata["source"]: doc.page_content
    for doc in soc_documents
}

for question in soc_goldset:
    if question["answerable"]:
        source = question["source"]
        for phrase in question["evidence"]:
            assert phrase.lower() in source_texts[source].lower(), (
                f"Missing evidence for {question['qid']}"
            )

print("\nOriginal knowledge base verified!")


# ------------------------------------------------------------
# Step 5: Split documents using LangChain
# ------------------------------------------------------------

# Original RAG baseline:
# Chunk size = 600 characters
# Overlap = 100 characters
# Retrieved chunks = 4
# Maximum cosine distance = 0.75

SOC_CHUNK_SIZE = 600
SOC_OVERLAP = 100

soc_splitter = CharacterTextSplitter(
    separator="",
    chunk_size=SOC_CHUNK_SIZE,
    chunk_overlap=SOC_OVERLAP,
    length_function=len
)

soc_chunks = soc_splitter.split_documents(soc_documents)


# ------------------------------------------------------------
# Step 6: Verify the chunks and their metadata
# ------------------------------------------------------------

print("\nTotal chunks created:", len(soc_chunks))

chunk_counts = Counter(
    doc.metadata["source"] for doc in soc_chunks
)

for filename, count in sorted(chunk_counts.items()):
    print(f"{filename}: {count} chunks")

print("\nFirst chunk preview:")
print(soc_chunks[0].page_content[:200])

print("\nFirst chunk metadata:")
print(soc_chunks[0].metadata)


Documents loaded: 5
 - email_authentication.md
 - incident_response.md
 - malware_analysis.md
 - phishing_triage.md
 - security_awareness.md

Evaluation questions: 8
Answerable: 6
Unanswerable: 2

Original knowledge base verified!

Total chunks created: 30
email_authentication.md: 6 chunks
incident_response.md: 6 chunks
malware_analysis.md: 6 chunks
phishing_triage.md: 6 chunks
security_awareness.md: 6 chunks

First chunk preview:
# Fictional SOC Notes: Email Authentication Checks

## Purpose
These original teaching notes explain how a fictional SOC uses email-authentication signals during suspicious-message review. Authenticat

First chunk metadata:
{'source': 'email_authentication.md'}


In [11]:

# ============================================================
# PART 8B — Custom Chroma Vector Store and Retriever
# IT7075: Applied AI for Cybersecurity
# ============================================================

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


# ------------------------------------------------------------
# Step 1: Initialize the same embedding model as Mini-Project 2
# ------------------------------------------------------------

soc_embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)


# ------------------------------------------------------------
# Step 2: Create the custom Chroma vector store
# ------------------------------------------------------------

# Use cosine distance to match the original RAG implementation.
# Stable chunk IDs prevent duplicates when this cell is rerun.

soc_db = Chroma.from_documents(
    documents=soc_chunks,
    embedding=soc_embeddings,
    ids=[f"soc-chunk-{i:04d}" for i in range(len(soc_chunks))],
    collection_name="it7075_soc_rag",
    collection_metadata={"hnsw:space": "cosine"}
)

print("Chunks stored:", soc_db._collection.count())


# ------------------------------------------------------------
# Step 3: Configure the baseline retriever
# ------------------------------------------------------------

# Original baseline: k=4, maximum cosine distance=0.75.
# For cosine relevance = 1 - distance, threshold = 0.25.

SOC_K = 4
SOC_SCORE_THRESHOLD = 0.25

soc_retriever = soc_db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": SOC_K,
        "score_threshold": SOC_SCORE_THRESHOLD
    }
)

print("Retriever configured")
print("k:", SOC_K)
print("Minimum relevance:", SOC_SCORE_THRESHOLD)


# ------------------------------------------------------------
# Step 4: Test retrieval with original gold-set questions
# ------------------------------------------------------------

test_cases = [
    soc_goldset[0],   # Answerable question
    soc_goldset[-1]   # Unanswerable budget question
]

for test in test_cases:
    question = test["question"]
    retrieved = soc_retriever.invoke(question)

    print("\nQuestion:", question)
    print("Expected answerable:", test["answerable"])
    print("Retrieved chunks:", len(retrieved))

    for i, doc in enumerate(retrieved, start=1):
        print(
            f"  {i}. Source: {doc.metadata.get('source')}"
        )
        print(
            "     Preview:",
            doc.page_content[:120].replace("\n", " "),
            "..."
        )


# ------------------------------------------------------------
# Step 5: Inspect actual cosine relevance scores
# ------------------------------------------------------------

print("\nTop relevance scores:")

for test in test_cases:
    scored_docs = soc_db.similarity_search_with_relevance_scores(
        test["question"],
        k=2
    )

    print("\n", test["qid"], test["question"])

    for doc, score in scored_docs:
        print(
            f"  Score: {score:.3f} | "
            f"Source: {doc.metadata.get('source')}"
        )


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2688.29it/s]


Chunks stored: 30
Retriever configured
k: 4
Minimum relevance: 0.25

Question: What details should an analyst record before classifying a reported phishing email?
Expected answerable: True
Retrieved chunks: 4
  1. Source: phishing_triage.md
     Preview: # Fictional SOC Runbook: Phishing Email Triage  ## Purpose and scope This runbook is an original educational simulation  ...
  2. Source: phishing_triage.md
     Preview: ous email, the analyst creates a case using the report timestamp and a unique case identifier. The analyst should preser ...
  3. Source: phishing_triage.md
     Preview: as informational, suspicious, or confirmed malicious based on documented indicators. If a user entered credentials or ap ...
  4. Source: security_awareness.md
     Preview: loyees should report suspicious messages using the approved phishing-reporting button or security help channel, not forw ...

Question: What is the exact annual dollar budget for the fictional SOC security awareness program?
Expe

In [12]:

# ============================================================
# PART 8C — Custom SOC RAG Prompt and LangChain Chain
# IT7075: Applied AI for Cybersecurity
# ============================================================

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough


# ------------------------------------------------------------
# Step 1: Format the retrieved cybersecurity documents
# ------------------------------------------------------------

def format_soc_docs(docs):
    """Combine retrieved chunks and preserve their source names."""

    return "\n\n".join(
        f"Source: {doc.metadata.get('source', 'unknown')}\n"
        f"{doc.page_content}"
        for doc in docs
    )


# ------------------------------------------------------------
# Step 2: Create the custom cybersecurity prompt template
# ------------------------------------------------------------

soc_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a cybersecurity SOC knowledge-base assistant. "
        "Answer questions using ONLY the retrieved document context. "
        "Do not invent facts, procedures, dollar amounts, or policies. "
        "If the context explicitly supports an answer, begin with [RAG] "
        "and identify the supporting source document. "
        "If the context does not provide enough evidence to answer, "
        "respond with [LLM] I don't know -- the knowledge base "
        "does not provide that information. "
        "Never treat a related topic as proof of a specific fact.\n\n"
        "# Context\n{context}"
    ),
    (
        "human",
        "{question}"
    )
])


# ------------------------------------------------------------
# Step 3: Assemble the LangChain RAG pipeline
# ------------------------------------------------------------

# Reuse 'model' from TODO 5.
# It already includes the output parser or offline fallback.

soc_chain = (
    {
        "context": soc_retriever | RunnableLambda(format_soc_docs),
        "question": RunnablePassthrough()
    }
    | soc_prompt
    | model
)


# ------------------------------------------------------------
# Step 4: Test the completed SOC pipeline
# ------------------------------------------------------------

print("Custom SOC LangChain chain created successfully!")

test_question = soc_goldset[1]["question"]

print("\nQuestion:", test_question)
print("\nResponse:")
print(soc_chain.invoke(test_question)[:600])


Custom SOC LangChain chain created successfully!

Question: Which email authentication policies or dispositions does DMARC commonly describe?

Response:
[RAG] (offline mode) From the retrieved context: Source: email_authentication.md
g domain and whether verification passed, failed, or was unavailable. A failed signature can have benign explanations, such as content modification, so it should be interpreted alongside o


### Answer your questions and measure the result

In [13]:

# ============================================================
# PART 8D — Test the Custom SOC RAG System
# IT7075: Applied AI for Cybersecurity
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# Step 1: Ask all eight original evaluation questions
# ------------------------------------------------------------

soc_test_results = []

for item in soc_goldset:

    question = item["question"]

    # Retrieve evidence using the LangChain retriever
    docs = soc_retriever.invoke(question)

    # Generate the answer using the completed LangChain chain
    response = soc_chain.invoke(question)

    # Check whether expected evidence appears in retrieved text
    retrieved_text = "\n".join(
        doc.page_content for doc in docs
    ).lower()

    evidence_found = (
        any(
            phrase.lower() in retrieved_text
            for phrase in item["evidence"]
        )
        if item["answerable"]
        else False
    )

    # Identify whether the answer component refused
    refused = response.strip().startswith("[LLM]")

    soc_test_results.append({
        "qid": item["qid"],
        "question": question,
        "answerable": item["answerable"],
        "chunks": len(docs),
        "sources": ", ".join(sorted(set(
            doc.metadata.get("source", "unknown")
            for doc in docs
        ))),
        "evidence_found": evidence_found,
        "refused": refused,
        "response": response
    })


# ------------------------------------------------------------
# Step 2: Print the evaluation summary
# ------------------------------------------------------------

summary = pd.DataFrame(soc_test_results)

print("CUSTOM SOC LANGCHAIN RAG — EVALUATION SUMMARY\n")

print(summary[
    ["qid", "answerable", "chunks", "evidence_found", "refused"]
].to_string(index=False))


# ------------------------------------------------------------
# Step 3: Calculate preliminary performance
# ------------------------------------------------------------

answerable_results = summary[summary["answerable"] == True]
unanswerable_results = summary[summary["answerable"] == False]

found = int(answerable_results["evidence_found"].sum())
refused = int(unanswerable_results["refused"].sum())

print("\nEvidence found:", found, "/ 6")
print("Refusals held:", refused, "/ 2")


# ------------------------------------------------------------
# Step 4: Display a readable demonstration transcript
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CUSTOM SOC RAG — QUESTION AND ANSWER TRANSCRIPT")
print("=" * 60)

# Show a supported question and both unsupported questions
for qid in ["Q2", "Q7", "Q8"]:

    result = next(
        row for row in soc_test_results
        if row["qid"] == qid
    )

    print(f"\n{qid}: {result['question']}")
    print("Expected answerable:", result["answerable"])
    print("Retrieved chunks:", result["chunks"])
    print("Sources:", result["sources"] or "None")
    print("Response:")
    print(result["response"][:500])
    print("-" * 60)


CUSTOM SOC LANGCHAIN RAG — EVALUATION SUMMARY

qid  answerable  chunks  evidence_found  refused
 Q1        True       4            True    False
 Q2        True       4            True    False
 Q3        True       4            True    False
 Q4        True       4            True    False
 Q5        True       4            True    False
 Q6        True       4            True    False
 Q7       False       0           False     True
 Q8       False       4           False    False

Evidence found: 6 / 6
Refusals held: 1 / 2

CUSTOM SOC RAG — QUESTION AND ANSWER TRANSCRIPT

Q2: Which email authentication policies or dispositions does DMARC commonly describe?
Expected answerable: True
Retrieved chunks: 4
Sources: email_authentication.md
Response:
[RAG] (offline mode) From the retrieved context: Source: email_authentication.md
g domain and whether verification passed, failed, or was unavailable. A failed signature can have benign explanations, such as content modification, so it should 

In [14]:

# PART 9A — Verify cosine distance and relevance scores

question = soc_goldset[0]["question"]

distance = soc_db.similarity_search_with_score(
    question, k=1
)[0][1]

relevance = soc_db.similarity_search_with_relevance_scores(
    question, k=1
)[0][1]

print(f"Cosine distance: {distance:.4f}")
print(f"LangChain relevance: {relevance:.4f}")
print(f"1 - distance: {1 - distance:.4f}")

assert abs(relevance - (1 - distance)) < 0.001

print("PASS: Threshold conversion verified!")


Cosine distance: 0.4266
LangChain relevance: 0.5734
1 - distance: 0.5734
PASS: Threshold conversion verified!


In [15]:

# ============================================================
# PART 9B — LangChain RAG Experimental Evaluation
# IT7075: Applied AI for Cybersecurity
# ============================================================

import pandas as pd
from pathlib import Path
from langchain_chroma import Chroma
from langchain_text_splitters import CharacterTextSplitter


# ------------------------------------------------------------
# Step 1: Define the seven original experiment settings
# ------------------------------------------------------------

experiments = [
    ("Baseline", 600, 4, 0.75),
    ("Small chunks", 300, 4, 0.75),
    ("Large chunks", 900, 4, 0.75),
    ("Fewer hits", 600, 2, 0.75),
    ("More hits", 600, 6, 0.75),
    ("Strict threshold", 600, 4, 0.50),
    ("Loose threshold", 600, 4, 0.90)
]

OVERLAP = 100

# Reuse the vector store from Part 8B for 600-character chunks.
experiment_stores = {600: soc_db}


# ------------------------------------------------------------
# Step 2: Build vector stores for alternative chunk sizes
# ------------------------------------------------------------

def get_experiment_store(chunk_size):

    if chunk_size not in experiment_stores:

        splitter = CharacterTextSplitter(
            separator="",
            chunk_size=chunk_size,
            chunk_overlap=OVERLAP,
            length_function=len
        )

        test_chunks = splitter.split_documents(soc_documents)

        store = Chroma.from_documents(
            documents=test_chunks,
            embedding=soc_embeddings,
            ids=[
                f"experiment-{chunk_size}-{i:04d}"
                for i in range(len(test_chunks))
            ],
            collection_name=f"it7075_soc_exp_{chunk_size}",
            collection_metadata={"hnsw:space": "cosine"}
        )

        experiment_stores[chunk_size] = store

    return experiment_stores[chunk_size]


# ------------------------------------------------------------
# Step 3: Score evidence coverage and retrieval refusals
# ------------------------------------------------------------

def score_soc_retriever(retriever, goldset):

    found = 0
    refused = 0

    for question in goldset:

        docs = retriever.invoke(question["question"])

        if question["answerable"]:

            # Match the original RAG scoring rule:
            # Any complete evidence phrase in any retrieved chunk.
            evidence_match = any(
                phrase.lower() in doc.page_content.lower()
                for doc in docs
                for phrase in question["evidence"]
            )

            if evidence_match:
                found += 1

        else:
            # Same refusal rule as the original project.
            if len(docs) == 0:
                refused += 1

    return found, refused


# ------------------------------------------------------------
# Step 4: Run the experiments
# ------------------------------------------------------------

experiment_rows = []

for name, chunk_size, k, max_distance in experiments:

    store = get_experiment_store(chunk_size)

    # Convert original cosine distance into minimum relevance.
    threshold = round(1.0 - max_distance, 2)

    test_retriever = store.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={
            "k": k,
            "score_threshold": threshold
        }
    )

    found, refused = score_soc_retriever(
        test_retriever,
        soc_goldset
    )

    experiment_rows.append({
        "configuration": name,
        "chunk_size": chunk_size,
        "overlap": OVERLAP,
        "k": k,
        "max_distance": max_distance,
        "score_threshold": threshold,
        "found": found,
        "refused": refused
    })

    print(f"{name}: Evidence {found}/6, Refusals {refused}/2")


# ------------------------------------------------------------
# Step 5: Save final LangChain results
# ------------------------------------------------------------

results_langchain = pd.DataFrame(experiment_rows)

results_langchain.to_csv(
    "results_langchain.csv",
    index=False
)

print("\n=== LANGCHAIN EXPERIMENT RESULTS ===")
print(results_langchain.to_string(index=False))

print("\nSaved results_langchain.csv")


# ------------------------------------------------------------
# Step 6: Compare with the original RAG results
# ------------------------------------------------------------

old_results_file = None

for candidate in [
    Path("../rag_project/results.csv"),
    Path("../results.csv")
]:
    if candidate.exists():
        old_results_file = candidate
        break

if old_results_file is not None:

    old_results = pd.read_csv(old_results_file)

    old_summary = old_results[
        ["configuration", "found", "refused"]
    ].rename(columns={
        "found": "original_found",
        "refused": "original_refused"
    })

    new_summary = results_langchain[
        ["configuration", "found", "refused"]
    ].rename(columns={
        "found": "langchain_found",
        "refused": "langchain_refused"
    })

    comparison = old_summary.merge(
        new_summary,
        on="configuration"
    )

    print("\n=== ORIGINAL RAG VS LANGCHAIN ===")
    print(comparison.to_string(index=False))

else:
    print("\nOriginal results.csv not found.")
    print("LangChain results were saved successfully.")


Baseline: Evidence 6/6, Refusals 1/2
Small chunks: Evidence 5/6, Refusals 1/2
Large chunks: Evidence 6/6, Refusals 1/2
Fewer hits: Evidence 5/6, Refusals 1/2
More hits: Evidence 6/6, Refusals 1/2
Strict threshold: Evidence 6/6, Refusals 1/2
Loose threshold: Evidence 6/6, Refusals 1/2

=== LANGCHAIN EXPERIMENT RESULTS ===
   configuration  chunk_size  overlap  k  max_distance  score_threshold  found  refused
        Baseline         600      100  4          0.75             0.25      6        1
    Small chunks         300      100  4          0.75             0.25      5        1
    Large chunks         900      100  4          0.75             0.25      6        1
      Fewer hits         600      100  2          0.75             0.25      5        1
       More hits         600      100  6          0.75             0.25      6        1
Strict threshold         600      100  4          0.50             0.50      6        1
 Loose threshold         600      100  4          0.90       

---

## What to write up

One PDF report and a video of five to ten minutes. See `MiniProject_FrameworkRAG.md` for the full
requirements. The questions that matter:

- What did LangChain replace? Name the code from your RAG-module notebook that each LangChain object
  made unnecessary. Where is that a genuine win, and where does it hide something you would rather see?
- On the same documents and the same questions, how do the two systems compare on the numbers you
  measured, and how much work did each take to build?
- Which question is hardest for this version, and what does it retrieve instead?
- The `score_threshold` here is a relevance score, not the distance you used in the RAG module. Higher
  means more similar. Why does that flip the comparison, and what happens if you get it backwards?

Submit one PDF report with the link to your project folder and the link to your video on its first page.
The notebook, your documents, `goldset.json`, and your results live in the repository, which is private,
with your instructor and the TA added as collaborators. The notebook must be committed with its outputs
showing.